# OMR corner-marker detector — Colab GPU training

Trains a single-class (`marker_corners`) YOLOv8 detector to localize the 4 sheet corner markers, so YOLO can drive the warp directly (skip C++ blob detection).

**Before running:** package the dataset locally with `python scripts/package_corner_dataset.py` — it writes `corner_dataset.zip` into `G:\My Drive\omr_corner_train\`, which syncs to Drive. Each time you add more CVAT labels, re-run that script and re-run this notebook (full retrain on the bigger set — YOLO does not learn incrementally).

Runtime → Change runtime type → **GPU (T4)**.

In [1]:
# 1. Mount Drive + check GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi -L

Mounted at /content/drive
GPU 0: Tesla T4 (UUID: GPU-e2c3a612-2174-3a4b-9bd4-24bb5479a233)


In [2]:
# 2. Install
!pip -q install ultralytics onnx onnxslim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 92.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 22.4 MB/s eta 0:00:00


In [3]:
# 3. Unzip dataset from Drive to fast local disk + write data.yaml
import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/omr_corner_train'   # adjust if your folder differs
ZIP = os.path.join(DRIVE_DIR, 'corner_dataset.zip')
DST = '/content/corner_dataset'
assert os.path.exists(ZIP), f'missing {ZIP} — run scripts/package_corner_dataset.py first'
if os.path.exists(DST):
    import shutil; shutil.rmtree('/content/corner_dataset', ignore_errors=True)
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content')
with open(os.path.join(DST, 'data.yaml'), 'w') as f:
    f.write(f'path: {DST}\ntrain: images/train\nval: images/val\nnames:\n  0: marker_corners\n')
ntr = len(os.listdir(os.path.join(DST, 'images/train')))
nva = len(os.listdir(os.path.join(DST, 'images/val')))
print(f'train={ntr}  val={nva}')

train=158  val=28


In [4]:
# 4. Train (GPU). Corners are tiny (~1.8% of width) -> keep imgsz high.
from ultralytics import YOLO
IMGSZ = 960
model = YOLO('yolov8n.pt')
model.train(
    data=f'{DST}/data.yaml', epochs=200, imgsz=IMGSZ, batch=16, device=0,
    patience=40, project='/content/runs', name='corners', exist_ok=True,
    fliplr=0.0, flipud=0.0, mosaic=0.0, degrees=0.0, scale=0.2, plots=True)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/corner_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f80f40bc4d0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [5]:
# 5. Eval corner localization on val (top-4 boxes strategy; we always expect 4 corners)
import glob, math, statistics as st
from ultralytics import YOLO
W = '/content/runs/corners/weights/best.pt'
model = YOLO(W)

def sort_corners(pts):
    cy = sum(p[1] for p in pts)/len(pts)
    top = sorted([p for p in pts if p[1] < cy], key=lambda p: p[0])
    bot = sorted([p for p in pts if p[1] >= cy], key=lambda p: p[0])
    if len(top)!=2 or len(bot)!=2:
        s = sorted(pts, key=lambda p:(p[1],p[0])); top,bot = sorted(s[:2],key=lambda p:p[0]), sorted(s[2:],key=lambda p:p[0])
    return [top[0], top[1], bot[1], bot[0]]

def gt(stem, W_, H_):
    pts=[]
    for line in open(f'{DST}/labels/val/{stem}.txt'):
        c=line.split()
        if len(c)>=5: pts.append((float(c[1])*W_, float(c[2])*H_))
    return pts

full, errs = 0, []
imgs = sorted(glob.glob(f'{DST}/images/val/*.jpg'))
for ip in imgs:
    stem = os.path.splitext(os.path.basename(ip))[0]
    r = model.predict(ip, imgsz=IMGSZ, conf=0.01, verbose=False)[0]
    Wd, Hd = r.orig_shape[1], r.orig_shape[0]
    dets = sorted([((b[0]+b[2])/2,(b[1]+b[3])/2,float(c)) for b,c in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist())], key=lambda d:-d[2])
    if len(dets) >= 4:
        full += 1
        pred = sort_corners([(d[0],d[1]) for d in dets[:4]])
        g = sort_corners(gt(stem, Wd, Hd))
        errs += [math.hypot(p[0]-q[0], p[1]-q[1]) for p,q in zip(pred,g)]
print(f'recovered >=4 corners: {full}/{len(imgs)} ({100*full/len(imgs):.0f}%)')
if errs:
    print(f'per-corner px error: median={st.median(errs):.1f}  p90={sorted(errs)[int(.9*len(errs))]:.1f}  max={max(errs):.1f}')

recovered >=4 corners: 28/28 (100%)
per-corner px error: median=1.2  p90=2.4  max=4.8


In [6]:
# 6. Export ONNX (for onnxruntime-web) + copy artifacts back to Drive
import shutil, datetime
model = YOLO(W)
onnx_path = model.export(format='onnx', imgsz=IMGSZ, opset=12, simplify=True)
out = os.path.join(DRIVE_DIR, 'out_' + datetime.datetime.now().strftime('%Y%m%d_%H%M'))
os.makedirs(out, exist_ok=True)
for src in [W, str(onnx_path), '/content/runs/corners/results.png', '/content/runs/corners/results.csv']:
    if os.path.exists(src): shutil.copy2(src, out)
print('artifacts ->', out)
print(os.listdir(out))
# NOTE: production web YOLO uses 640 input. If you integrate this model, either
# re-export at imgsz=640 (markers ~11px, verify recall) or bump the web input size.

Ultralytics 8.4.70 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/corners/weights/best.pt' with input shape (1, 3, 960, 960) BCHW and output shape(s) (1, 5, 18900) (6.0 MB)
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 295ms
Prepared 1 package in 376ms
Installed 1 package in 8ms
 + onnxruntime==1.27.0

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 12...
ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 2.4s, saved as '/content/runs/corners/weights/best.onnx' (11.9 MB)